In [ ]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 11, Publication-ready output

Companion notebook to [11_publication_output.md](11_publication_output.md). `chain_to_latex` / `chain_to_tikz` and their `_document` wrappers turn a `ProofChain` into LaTeX you can paste into a paper or compile into a standalone PDF.

## A worked chain

Tiny but real proof: `d(d(ω)) == 0`. Two steps, enough to exercise the renderers without flooding the output.

In [ ]:
from jacopy.algebra.derivation import Act
from jacopy.calculus.exterior_d import d
from jacopy.core.expr import Symbol, Integer
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.proof import prove_equivalence

reg = PropertyRegistry()
omega = Symbol("ω"); reg.declare(omega, Graded(degree=2))

chain = prove_equivalence(
    Act(d, Act(d, omega)),
    Integer(0),
    registry=reg,
)
print(f"steps: {len(chain)}")
for s in chain.steps:
    print(f"  [{s.rule}] {s.before} → {s.after}")

## Inline LaTeX

`chain_to_latex(chain)` returns a `gather*` block, one math row per step, wrapped in `\allowdisplaybreaks\scriptsize` so long chains page-break properly. Paste straight into any `.tex` source that loads `amsmath`.

In [ ]:
from jacopy.display import chain_to_latex

body = chain_to_latex(chain)
print(body)

Each row uses `\to` (not `=`), explicit rewriting direction. The `\quad \text{[rule] (provenance)}` annotation reads as the step's justification.

## Standalone document

`chain_to_latex_document` adds a full `\documentclass{article}` preamble plus optional `\title`/`\author`/`\maketitle`. Write to disk and `pdflatex out.tex` compiles directly.

In [ ]:
from jacopy.display import chain_to_latex_document

doc = chain_to_latex_document(
    chain,
    title="d² = 0",
    author="jacopy",
)
print(doc[:400])
print("...")
print(doc[-80:])

`preamble_extras="..."` splices project-specific macros (e.g. `\input{macros.tex}`) between the default preamble and `\begin{document}`.

## TikZ diagram

`chain_to_tikz` renders the same chain as a vertical `tikzpicture`. Each `before`/`after` becomes a boxed node, `n + 1` nodes for `n` steps, with arrows labelled by rule name and provenance tag.

In [ ]:
from jacopy.display import chain_to_tikz

diagram = chain_to_tikz(chain)
print(diagram)

## Standalone TikZ document

`chain_to_tikz_document` wraps the diagram in a full document with the `tikz` + `positioning` preamble.

In [ ]:
from jacopy.display import chain_to_tikz_document

doc = chain_to_tikz_document(
    chain,
    title="d² = 0 (diagram)",
    node_distance="1.4cm",
)
print(doc[:400])

`node_distance` tunes vertical spacing, bump it up when expression labels crowd each other.

## Round-trip to PDF (optional)

Both `_document` outputs compile with any LaTeX engine. The cell below is illustrative, skip executing it unless `pdflatex` is on your PATH.

In [ ]:
import shutil

if shutil.which("pdflatex") is None:
    print("pdflatex not found on PATH, skipping the round-trip demo.")
else:
    import subprocess, tempfile
    from pathlib import Path
    with tempfile.TemporaryDirectory() as tmp:
        src = Path(tmp) / "proof.tex"
        src.write_text(doc, encoding="utf-8")
        subprocess.run([
            "pdflatex", "-interaction=nonstopmode", str(src),
        ], cwd=tmp, check=True, capture_output=True)
        print(sorted(p.name for p in Path(tmp).iterdir()))

## Summary

* Four helpers, two inline (`chain_to_latex`, `chain_to_tikz`), two standalone (`chain_to_latex_document`, `chain_to_tikz_document`).
* Inline: paste into existing `.tex`. Standalone: write + `pdflatex`.
* `title` / `author` / `preamble_extras` are kwargs on the `_document` variants, empty strings produce a body-only document.
* Nested sub-proofs flatten in both renderers; compose by hand if you want a tree.